# Setup


In [ ]:
import sys
from pathlib import Path

# import os
import importlib

sys.path.append(str(Path.cwd().parent.parent.parent.absolute()))

import config

importlib.reload(config)

import argparse
from context import Context
from utils.utils_chain import WrapperAddress
from tools.runners.farm_runner import (
    upgrade_farmv13unlocked_contracts,
    get_all_farm_v13unlocked_addresses,
    fetch_and_save_farms_from_chain,
    resume_farmv13unlocked_contracts,
)

# os.environ["MX_DEX_ENV"] = "shadowfork4"

context = Context()

# TODO: fill in the legacy farm v1.3 (unlocked rewards) release wasm and its code hash
farm_v13_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v1.5.0-farm-unlocked-exit-only/farm.wasm"
farm_v13_code_hash = "436d641c70728d179234a23c20afe6f2d3f9fd09367697d9ddcfb6cd33fe92ca"
assert farm_v13_wasm and farm_v13_code_hash, "Set the farm v1.3 wasm and its code hash"

Check owner balance for required minimum


In [ ]:
current_balance = context.network_provider.proxy.get_account(
    context.deployer_account.address
).balance
assert current_balance > 20 * 10**18, "Deployer account doesn't have enough balance"

Clean outputs folder


In [ ]:
import shutil

if config.UPGRADER_OUTPUT_FOLDER.exists():
    shutil.rmtree(config.UPGRADER_OUTPUT_FOLDER)

Fetch legacy farm contracts


In [ ]:
fetch_and_save_farms_from_chain("")

farm_addresses = get_all_farm_v13unlocked_addresses()
print(f"Found {len(farm_addresses)} v1.3 farm contracts with unlocked rewards:")
for address in farm_addresses:
    print(address)

# Upgrade farm v1.3 contracts with unlocked rewards


In [ ]:
args = argparse.Namespace(bytecode=farm_v13_wasm, compare_states=True)
upgrade_farmv13unlocked_contracts(args)

farm_addresses = get_all_farm_v13unlocked_addresses()
print(f"Checking {len(farm_addresses)} farm contracts for correct code hash...")
for address in farm_addresses:
    code_hash = context.network_provider.proxy.get_account(
        WrapperAddress(address)
    ).contract_code_hash.hex()
    assert code_hash == farm_v13_code_hash, f"Code hash mismatch for {address}"
print("Done!")

# Resume contracts

In [ ]:
resume_farmv13unlocked_contracts("")